### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="obesity_estimation",
    dataset_year="2019",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5H31Z",
    download_description="""
wget https://archive.ics.uci.edu/static/public/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition.zip && unzip estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition.zip && rm estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition.zip
mkdir -p local-data-warehouse/obesity_estimation && mv *.csv local-data-warehouse/obesity_estimation/
""",
    # References
    academic_reference_bibtex=r"""@article{palechor2019dataset,
    title={Dataset for estimation of obesity levels based on eating habits and physical condition in individuals from Colombia, Peru and Mexico},
    author={Palechor, Fabio Mendoza and De la Hoz Manotas, Alexis},
    journal={Data in brief},
    volume={25},
    pages={104344},
    year={2019},
    publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="palechor2019dataset",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We use only the real data, throwing away the SMOTE samples.
- We make the task a regression task, re-creating the body mass values underlying the discretized obesity levels, and drop weight and height which leak the target partially. Thus, we create a regression task of estimating the body mass of individuals based on their eating habits and other questions from the questionnaire. The original task is trivial to solve due to the leak and thus not interesting.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="BodyMass",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "ObesityDataSet_raw_and_data_sinthetic.csv")

# Filter to only real data (real data is int and not float values, see original .csv file)
df = df.iloc[:498]

# Creat real target
df["BodyMass"] = df["Weight"] / (df["Height"] ** 2)
df = df.drop(columns=["Height", "Weight", "NObeyesdad"])

as_cat_type = [
    "Gender", "family_history_with_overweight", "FAVC", "FCVC",
    "CAEC", "SMOKE", "SCC", "CALC", "MTRANS",
    # Numerical on UCI, but from the paper we know this was categorial in the questionnaire:
    "CH2O", "FAF", "TUE", "NCP",
]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()